# Parquet Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import sys, logging, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py
import pyarrow as pa
import pyarrow.parquet as pq
import logging
import awkward as ak
import uproot

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

from pathlib import Path
import logging

## Uproot testing

Stable sim output, 64 events, single muon, 1 GeV

In [8]:
stable_edm4hep_path = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/single_particle_pilot/single_muon_1GeV/v2/runs/0/edm4hep.root"
stable_file = uproot.open(stable_edm4hep_path)
print(f"Keys: {stable_file.keys()}")
events = stable_file["events"]
print(f"Events keys: {events.keys()}")

Keys: ['events;1', 'runs;1', 'meta;1', 'metadata;1', 'podio_metadata;1']
Events keys: ['ECalBarrelCollection', 'ECalBarrelCollection/ECalBarrelCollection.cellID', 'ECalBarrelCollection/ECalBarrelCollection.energy', 'ECalBarrelCollection/ECalBarrelCollection.position.x', 'ECalBarrelCollection/ECalBarrelCollection.position.y', 'ECalBarrelCollection/ECalBarrelCollection.position.z', 'ECalBarrelCollection/ECalBarrelCollection.contributions_begin', 'ECalBarrelCollection/ECalBarrelCollection.contributions_end', '_ECalBarrelCollection_contributions', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.index', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.collectionID', 'ECalBarrelCollectionContributions', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.PDG', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.energy', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.time', 'ECalBarrelColl

In [9]:
events

<TTree 'events' (41 branches) at 0x7fe75a4c8590>

In [10]:
events.num_entries

64

New sim output, 64 events, single muon, 1 GeV

In [11]:
stable_edm4hep_path = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/single_particle_pilot/single_muon_1GeV/v3/runs/0/edm4hep.root"
stable_file = uproot.open(stable_edm4hep_path)
print(f"Keys: {stable_file.keys()}")
events = stable_file["events"]
print(f"Events keys: {events.keys()}")

Keys: ['events;1', 'metadata;1', 'runs;1', 'meta;1', 'podio_metadata;1']
Events keys: ['ECalBarrelCollection', 'ECalBarrelCollection/ECalBarrelCollection.cellID', 'ECalBarrelCollection/ECalBarrelCollection.energy', 'ECalBarrelCollection/ECalBarrelCollection.position.x', 'ECalBarrelCollection/ECalBarrelCollection.position.y', 'ECalBarrelCollection/ECalBarrelCollection.position.z', 'ECalBarrelCollection/ECalBarrelCollection.contributions_begin', 'ECalBarrelCollection/ECalBarrelCollection.contributions_end', '_ECalBarrelCollection_contributions', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.index', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.collectionID', 'ECalBarrelCollectionContributions', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.PDG', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.energy', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.time', 'ECalBarrelColl

In [15]:
events["EventHeader/EventHeader.eventNumber"].arrays()

<Array [{...}, {...}, {...}, ..., {...}, {...}] type='64 * {"EventHeader.ev...'>

In [13]:
events.num_entries

64

In [ ]:
edm4hep_path = run_dir / "edm4hep.root"
batch = EDM4hepEventBatch(str(edm4hep_path), events=local_events, condense_calo=False)

2025-10-27 06:18:15,670 - DEBUG - pyedm4hep.event_batch - EDM4hepEventBatch init: file=/global/cfs/cdirs/m4958/data/ColliderML/simulation/hard_scatter/ttbar/v1/runs/0/edm4hep.root, events=(0, 10), full_load=False, condense_calo=False


In [17]:
batch._ensure_loaded("tracker_hits")
batch._ensure_loaded("calo_hits")
batch._ensure_loaded("calo_contributions")

2025-10-27 06:18:21,891 - DEBUG - pyedm4hep.event_batch - _load_tracker_hits: start=0, stop=10
2025-10-27 06:18:21,893 - DEBUG - asyncio - Using selector: EpollSelector
2025-10-27 06:18:21,893 - DEBUG - asyncio - Using selector: EpollSelector
2025-10-27 06:18:22,024 - DEBUG - pyedm4hep.event_batch - tracker PixelBarrelReadout: rows=6661 cols=['event_id', 'subentry', 'cellID', 'time', 'x', 'y', 'z', 'detector']
2025-10-27 06:18:22,024 - DEBUG - pyedm4hep.event_batch - tracker PixelBarrelReadout: rows=6661 cols=['event_id', 'subentry', 'cellID', 'time', 'x', 'y', 'z', 'detector']
2025-10-27 06:18:22,065 - DEBUG - pyedm4hep.event_batch - tracker PixelEndcapReadout: rows=4943 cols=['event_id', 'subentry', 'cellID', 'time', 'x', 'y', 'z', 'detector']
2025-10-27 06:18:22,065 - DEBUG - pyedm4hep.event_batch - tracker PixelEndcapReadout: rows=4943 cols=['event_id', 'subentry', 'cellID', 'time', 'x', 'y', 'z', 'detector']
2025-10-27 06:18:22,092 - DEBUG - pyedm4hep.event_batch - tracker ShortSt

Any NAN values:  event_id       False
subentry       False
energy         False
time           False
particle_id    False
detector       False
hit_index      False
cellID         False
x              False
y              False
z              False
dtype: bool
